In [20]:
from datasets import load_dataset, DatasetDict, Dataset
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

In [21]:
grammar_data = load_dataset("jhu-clsp/jfleg")
# This dataset only include validation and test data

# Split validation data into train & validation data
split = grammar_data["validation"].train_test_split(test_size = 0.2, seed = 42)

dataset = DatasetDict({
    "train": split["train"], 
    "validation": split["test"], 
    "test": grammar_data["test"]
})

# Convert to pandas
grammar_train_df = dataset["train"].to_pandas()
grammar_val_df = dataset["validation"].to_pandas()
grammar_test_df = dataset["test"].to_pandas()

In [22]:
print(grammar_data)

DatasetDict({
    validation: Dataset({
        features: ['sentence', 'corrections'],
        num_rows: 755
    })
    test: Dataset({
        features: ['sentence', 'corrections'],
        num_rows: 748
    })
})


In [23]:
print("Training Data: ")
print(grammar_train_df.head())

Training Data: 
                                            sentence  \
0  The advertisemnet says that the car has space ...   
1  The government would like to reduce the popula...   
2  Also , the professor indicates prescribed fire...   
3  I believe , that the product 's quality mentio...   
4  As we all known when we were out something we ...   

                                         corrections  
0  [The advertisement says that the car has space...  
1  [The government would like to reduce the popul...  
2  [Also , the professor indicates that we should...  
3  [I believe that the product 's quality mention...  
4  [As we have all known , when we were out , som...  


In [24]:
print("Validation Data: ")
print(grammar_val_df.head())

Validation Data: 
                                            sentence  \
0  Furethermore the professor denies the reading ...   
1  Then , define your goals and work hard to achi...   
2  However , people do n't know there and they do...   
3  I think it 's harder for successful preson to ...   
4  The student must be capable of understanding i...   

                                         corrections  
0  [Furthermore the professor denies the reading ...  
1  [Then define your goals and work hard to achie...  
2  [However , people do n't know there and they d...  
3  [I think it 's harder for a successful person ...  
4  [The student must be capable of understanding ...  


In [25]:
print("Test Data: ")
print(grammar_test_df.head())

Test Data: 
                                            sentence  \
0  New and new technology has been introduced to ...   
1  One possible outcome is that an environmentall...   
2  Every person needs to know a bit about math , ...   
3  While the travel company will most likely show...   
4  Disadvantage is parking their car is very diff...   

                                         corrections  
0  [New technology has been introduced to society...  
1  [One possible outcome is that an environmental...  
2  [Every person needs to know a bit about math ,...  
3  [While the travel company will most likely sho...  
4  [A disadvantage is that parking their cars is ...  


In [26]:
# To get the whole available grammar dataset
grammar_data = pd.concat([grammar_train_df, grammar_val_df, grammar_test_df], ignore_index = True)

grammar_data = pd.DataFrame({
    "error": grammar_data["sentence"],
    "corrected": grammar_data["corrections"].apply(lambda x: x[0]),
    "error_type": "grammar"
})

In [27]:
spelling_pairs = []

with open("missp.dat", "r", encoding = "utf-8") as f:
    current_correct = None

    for line in f:
        # Remove whitespace from the beginning & end of string
        word = line.strip() 

        # To avoid empty line
        if not word: 
            continue

        if word.startswith("$"):
            # Remove $ from the word
            current_correct = word[1:]

        else: 
            spelling_pairs.append((word, current_correct))


In [28]:
# Convert spelling_pairs to DataFrame
spelling_data = pd.DataFrame(spelling_pairs, columns = ["error", "corrected"])
spelling_data["error_type"] = "spelling"

In [29]:
print(spelling_data.head())

      error corrected error_type
0        Ab    Albert   spelling
1   Ameraca   America   spelling
2   Amercia   America   spelling
3  Ameracan  American   spelling
4     Apirl     April   spelling


In [30]:
# Combine DataFrame
combined_df = pd.concat([grammar_data, spelling_data], ignore_index = True)

In [31]:
# Split data - train (80%), val (10%), test (10%)

# Split into train (80%) & temp (20%)
train_df, temp_df = train_test_split(combined_df, test_size = 0.2, random_state = 42, stratify = combined_df["error_type"])

# Split temp into val (10%) & test (10%)
val_df, test_df = train_test_split(temp_df, test_size = 0.5, random_state = 42, stratify = temp_df["error_type"])

In [32]:
print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))
print(train_df["error_type"].value_counts())

Train size: 30108
Validation size: 3764
Test size: 3764
error_type
spelling    28906
grammar      1202
Name: count, dtype: int64


In [33]:
# Convert to HuggingFace Datasets
train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)
test_ds = Dataset.from_pandas(test_df)

In [34]:
tokenizer = AutoTokenizer.from_pretrained("t5-small")

tokenized = tokenizer(
    combined_df["error"].tolist(),
    text_target = combined_df["corrected"].tolist(),
    padding = "max_length",
    truncation = True,
    max_length = 128, 
    return_tensors = "pt"
)

In [35]:
#print(tokenized)

def tokenize(example):
    return tokenizer(
        example["error"], 
        text_target = example["corrected"], 
        padding = "max_length", 
        truncation = True, 
        max_length = 128
    )

In [36]:
train_tokenized = train_ds.map(tokenize, batched = True)
val_tokenized = val_ds.map(tokenize, batched = True)
test_tokenized = test_ds.map(tokenize, batched = True)

Map: 100%|██████████| 3764/3764 [00:00<00:00, 6893.17 examples/s]


In [37]:
# Save preprocessed data into files
train_tokenized.save_to_disk("t5_tokenized_train")
val_tokenized.save_to_disk("t5_tokenized_val")
test_tokenized.save_to_disk("t5_tokenized_test")

Saving the dataset (1/1 shards): 100%|██████████| 3764/3764 [00:00<00:00, 185661.57 examples/s]


In [38]:
# Save tokenizer
tokenizer.save_pretrained("t5_tokenizer")

('t5_tokenizer\\tokenizer_config.json',
 't5_tokenizer\\special_tokens_map.json',
 't5_tokenizer\\tokenizer.json')